In [2]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

# if GPU is to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
import torch
torch.cuda.is_available()

True

In [5]:
def init():
    global env
    env = gym.make("LunarLander-v3", continuous=False, gravity=-10.0, enable_wind=False, wind_power=15.0, turbulence_power=1.5)

In [6]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))

 
class ReplayBuffer(object):
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, transition):
        self.buffer.append(transition)
    
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)
    
    def size(self):
        return len(self.buffer)


In [7]:
class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128).to(device)
        self.fc2 = nn.Linear(128, 64).to(device)
        self.fc3 = nn.Linear(64, output_dim).to(device)

    def forward(self, state):
        x = torch.relu(self.fc1(state)).to(device)
        x = torch.relu(self.fc2(x)).to(device)
        q_values = self.fc3(x).to(device)
        return q_values


In [8]:
init()

In [9]:
#!pip install swig
#!pip install gymnasium[box2d]

In [10]:
def select_action(agent, state):
    if random.random() < agent.epsilon:
        return agent.env.action_space.sample()  # Explore
    else:
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        q_values = agent.q_network(state_tensor)
        return torch.argmax(q_values).item()  # Exploit

In [11]:
class DQNAgent:
    def __init__(self, env):
        self.env = env
        self.input_dim = env.observation_space.shape[0]
        self.output_dim = env.action_space.n
        self.q_network = QNetwork(self.input_dim, self.output_dim)
        self.target_network = QNetwork(self.input_dim, self.output_dim)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=1e-3)
        self.replay_buffer = ReplayBuffer(10000)
        self.batch_size = 64
        self.gamma = 0.99
        self.epsilon = 0.1
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.update_target_network_frequency = 2000
        self.timestep = 0

    def select_action(self, state):
        if random.random() < self.epsilon:
            return self.env.action_space.sample()  # Explore
        else:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_network(state_tensor)
            return torch.argmax(q_values).item()  # Exploit

    def train(self):
        if self.replay_buffer.size() < self.batch_size:
            return
        
        # Sample a batch of experiences from the buffer
        batch = self.replay_buffer.sample(self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)

        # Compute Q-values
        current_q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        next_q_values = self.target_network(next_states).max(1)[0]
        target_q_values = rewards + (1 - dones) * self.gamma * next_q_values

        # Loss function (Mean Squared Error)
        loss = nn.MSELoss()(current_q_values, target_q_values)

        # Optimize the model
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Update epsilon (epsilon-greedy policy)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        # Update target network periodically
        self.timestep += 1
        if self.timestep % self.update_target_network_frequency == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())


In [12]:
# Create the environment
env = gym.make('LunarLander-v3')

# Initialize the agent
agent = DQNAgent(env)

# Training loop
num_episodes = 500
for episode in range(num_episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0
    
    while not done:
        # Select an action using epsilon-greedy policy
        action = agent.select_action(state)

        # Perform the action in the environment
        next_state, reward, done, truncated, _ = env.step(action)

        # Store the experience in the replay buffer
        agent.replay_buffer.push((state, action, reward, next_state, done))

        # Train the agent
        agent.train()

        # Update the total reward and state
        total_reward += reward
        state = next_state

        if(total_reward > 200): # this did not stop the training
            done = True
            env.close()

    print(f"Episode {episode + 1}/{num_episodes}, Total Reward: {total_reward}")

env.close()


Episode 1/500, Total Reward: -136.54488401071364


C:\Users\Bruger\AppData\Local\Temp\ipykernel_14524\1971712923.py:35: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  states = torch.FloatTensor(states).to(device)


Episode 2/500, Total Reward: -454.673697090131
Episode 3/500, Total Reward: -394.31471734594146
Episode 4/500, Total Reward: -239.93100972193488
Episode 5/500, Total Reward: -225.60162578013325
Episode 6/500, Total Reward: -614.1537342325696
Episode 7/500, Total Reward: -413.67298259447386
Episode 8/500, Total Reward: -639.5360721417677
Episode 9/500, Total Reward: -458.1255330305528
Episode 10/500, Total Reward: -277.64876958049445
Episode 11/500, Total Reward: -384.1193408258134
Episode 12/500, Total Reward: -309.68090754788795
Episode 13/500, Total Reward: -405.66286006039746
Episode 14/500, Total Reward: -239.13108608637998
Episode 15/500, Total Reward: -271.7621236267346
Episode 16/500, Total Reward: -193.73147044266526
Episode 17/500, Total Reward: -221.40340913937473
Episode 18/500, Total Reward: -283.68028014967314
Episode 19/500, Total Reward: -266.5539034030454
Episode 20/500, Total Reward: -228.13648678835008
Episode 21/500, Total Reward: -350.5792126315753
Episode 22/500, T

In [13]:
torch.save(agent.state_dict(), "lunar_lander_model.pth") # save model/agent

AttributeError: 'DQNAgent' object has no attribute 'state_dict'

In [ ]:
agent.load_state_dict(torch.load("lunar_lander_model.pth")) # load model
agent.to(device)

In [ ]:
# JC: added this part to render
# test
env_test = gym.make('LunarLander-v3', render_mode="human")

state, info = env_test.reset()
state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
done = False
while not done:
    action = agent.select_action(state)
    observation, reward, terminated, truncated, _ = env_test.step(action.item())
    reward = torch.tensor([reward], device=device)
    done = terminated or truncated
    if terminated:
        next_state = None
    else:
        next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
    
    state = next_state
    env_test.render()

env_test.close()

TypeError: select_action() missing 1 required positional argument: 'state'

: 

In [ ]:
# Evaluate the trained agent
env = gym.make('LunarLander-v3', render_mode="human") # it crashes when closing the window
state, _ = env.reset()
done = 0 # was simple boolean before
total_reward = 0

while done < 1000:
    done += 1
    action = agent.select_action(state)
    next_state, reward, done, truncated, _ = env.step(action)
    total_reward += reward
    state = next_state

print(f"Total Reward during evaluation: {total_reward}")


Total Reward during evaluation: 283.39186735090357


: 